# Reproduction of `Shared Lexical Task Heads`
Paper Link: https://arxiv.org/pdf/2604.22027

## Objectives:
- Replicate key findings from Table 1 (identifying Lexical Task Heads), Figure 3 (verifying head reuse across prompting styles), and Figure 4 (functional equivalence of heads via activation patching) on the Llama-3.1-8B-Instruct model.
- Extend the replication to the Llama-3.2 family to study how architectural scale affects the vertical localization and activation strength of shared task-representation circuits.

# Set Up

In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from typing import List, Dict, Set

# Standard Hugging Face and Interpretability tools
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    import transformer_lens
    from transformer_lens import HookedTransformer, utilities, patching
except ImportError:
    print("Installing transformer-lens for mechanistic interpretability...")
    !pip install transformer-lens -q
    import transformer_lens
    from transformer_lens import HookedTransformer, utilities, patching

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Installing transformer-lens for mechanistic interpretability...
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.1 MB/s eta 0:00:00
Using device: cuda


In [6]:
# from huggingface_hub import login
# login()

# Llama-3.1-8B-Instruct
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device_map="auto")

# Llama-3.2-1B-Instruct
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", device_map="auto")

messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


# Result 1 (Table 1): Identify Lexical Task Heads (LTH)

Lexical Task Heads (LTHs) are specialized attention heads that write human-interpretable task descriptions (e.g., "opposite" for an antonym task, or "plural" for a pluralization task) into the vocabulary space of the residual stream.

### Per-prompt LTH:
An attention head is classified as an LTH for a specific prompt if the task is actively verbalized in its output.
- Match Rule: At least $n = 1$ of the top $k = 10$ decoded tokens must match predefined list of task-descriptive terms.
- Tokenizer Handling: To prevent false negatives from Llama's Byte-Pair Encoder (BPE), the task-descriptive terms must include both raw strings and whitespace-prefixed variations (e.g., "opposite", " opposite", and "_opposite").

### Per-prompt-style LTH:
According to the paper criterion, a head is only finalized as an LTH for a prompting style if it consistently represents the task across the dataset.
- Consistency Threshold: The head must satisfy the Per-Prompt criterion on at least $p = 10\%$ of all prompts within that style for a target task.

## Country-Capital Task

In [7]:
country_capital_pairs = [
    {"country": "Japan", "capital": "Tokyo"},
    {"country": "Greece", "capital": "Athens"},
    {"country": "France", "capital": "Paris"},
    {"country": "Ecuador", "capital": "Quito"},
    {"country": "England", "capital": "London"},
    {"country": "Korea", "capital": "Seoul"},
    {"country": "Germany", "capital": "Berlin"},
    {"country": "Vietnam", "capital": "Hanoi"},
]

In [72]:
#  Instruction-based Prompting
instruction_templates = [
    "The capital city of {instance_input} is",
    "What is the capital of the country? Q: {instance_input}\nA:",
    "Country -> Capital Q: {instance_input}\nA:",
    "Task: given the country, answer the capital of the country. Q: {instance_input}\nA:",
    "What capital city does this country have? Q: {instance_input}\nA:",
    "Instructions: you are given a country and you need to output the capital city of that country. Q:\n{instance_input}\nA:"
]

# Example-based Prompting
example_templates = [
    "{query_country}:" # 0-shot
    "{demo_country1}: {demo_capital1}; {query_country}:" # 1-shot
    "{demo_country1}: {demo_capital1}; {demo_country2}: {demo_capital2}; {query_country}:" # 2-shot
    "{demo_country1}: {demo_capital1}; {demo_country2}: {demo_capital2}; {demo_country3}: {demo_capital3}; {demo_country4}: {demo_capital4}; {demo_country5}: {demo_capital5}; {query_country} " # 5-shot
]

In [78]:
import random

def generate_eval_prompts(pairs, style="instruction", template_idx=0, num_shots=5):
    eval_dataset = []

    for i, target in enumerate(pairs):
        query_country = target["country"]
        correct_answer = target["capital"]

        if style == "instruction":
            prompt = instruction_templates[template_idx].format(instance_input=query_country)
        elif style == "example":
            # Sample non-target pairs
            other_pairs = [p for j, p in enumerate(pairs) if j != i]
            demos = random.sample(other_pairs, num_shots)

            prompt_parts = []
            for d in demos:
                prompt_parts.append(f"{d['country']}: {d['capital']}")
            prompt_parts.append(f"{query_country}:")
            prompt = "; ".join(prompt_parts)

        eval_dataset.append({
            "prompt": prompt,
            "target": query_country,
            "correct_answer": correct_answer
        })

    return eval_dataset

inst_country_capital_eval_dataset = generate_eval_prompts(country_capital_pairs, style="instruction", template_idx=0)
exam_country_capital_eval_dataset = generate_eval_prompts(country_capital_pairs, style="example", template_idx=0, num_shots=2)

### Logit Lens

In [79]:
n_heads = model.config.num_attention_heads
n_layers = model.config.num_hidden_layers
n_embd = model.config.hidden_size
d_head = n_embd // n_heads

print(f"Number of heads: {n_heads}")
print(f"Number of layers: {n_layers}")
print(f"Embedding dimension: {n_embd}")
print(f"Head dimension: {d_head}")

k = 10 # Top k tokens to decode
n = 1 # Minimum matching tokens
p = 0.1 # p consistency filter across prompt style

TASK_DESCRIPTIVE_TERMS: Set[str] = {
    "city", "_city", "cities", "-city", "-capital", "Cities", " city", " cities", "capital", " capital", "City", "capitals", ".city", "Capital"
}

Number of heads: 32
Number of layers: 16
Embedding dimension: 2048
Head dimension: 64


In [80]:
# STEP 1: CACHE ACTIVATION HOOK
cached_attn_inputs = {}

def get_attn_hook(layer_idx: int):
  def hook(module, input, output):
    hidden_states = input[0]
    batch_size, seq_len, _ = hidden_states.shape
    # Reshape to isolate heads
    head_inputs = hidden_states.view(batch_size, seq_len, n_heads, d_head)
    # Store the last token's activations for each head
    cached_attn_inputs[layer_idx] = head_inputs[:, -1, :, :].detach().to(device)
  return hook

# Clear existing hooks to prevent duplicates if re-running
for layer_idx in range(n_layers):
  o_proj_module = model.model.layers[layer_idx].self_attn.o_proj
  o_proj_module._forward_hooks.clear()
  o_proj_module.register_forward_hook(get_attn_hook(layer_idx))

In [81]:
# STEP 2: APPLY LOGIT LENS
def get_head_logits(model, tokenizer, layer_idx: int, head_idx: int, head_input: torch.Tensor) -> torch.Tensor:
  device = model.device
  dtype = model.dtype
  head_input = head_input.to(device=device, dtype=dtype)

  o_proj_weight = model.model.layers[layer_idx].self_attn.o_proj.weight
  W_O_h = o_proj_weight[:, head_idx * d_head : (head_idx + 1) * d_head] # [n_embd, d_head]

  # Project head activation to residual stream
  # If head_input is [d_head], head_output becomes [n_embd]
  head_output = head_input @ W_O_h.t()

  final_ln = model.model.norm
  # Ensure we have a batch dimension for the LayerNorm
  normalized_output = final_ln(head_output.view(1, -1)) # [1, n_embd]

  # Project norm_output to vocabulary space
  lm_head = model.lm_head
  logits = lm_head(normalized_output) # [1, d_vocab]

  return logits

In [82]:
# STEP 3: RUN PIPELINE TO SEARCH LTH
def run_logit_lens_pipeline(eval_dataset: List[Dict[str, str]]) -> List[Dict]:
    match_matrix = torch.zeros((n_layers, n_heads, len(eval_dataset)), dtype=torch.bool)

    for prompt_idx, item in enumerate(eval_dataset):
      prompt = item["prompt"]
      correct_answer = item["correct_answer"]
      cached_attn_inputs.clear()

      inputs = tokenizer(prompt, return_tensors="pt").to(device)
      with torch.no_grad():
          outputs = model(**inputs)

      # Verify model prediction
      generated_logits = outputs.logits[:, -1, :]
      predicted_token_id = generated_logits.argmax(dim=-1)
      predicted_token = tokenizer.decode(predicted_token_id[0])

      if predicted_token.strip().lower() != correct_answer.lower():
        continue

      for layer_idx in range(n_layers):
        if layer_idx not in cached_attn_inputs: continue
        head_activations = cached_attn_inputs[layer_idx] # [batch, heads, d_head]

        for head_idx in range(n_heads):
          single_head_input = head_activations[0, head_idx, :] # [d_head]
          logits = get_head_logits(model, tokenizer, layer_idx, head_idx, single_head_input)

          # Fix: Ensure indices are extracted as a list to avoid 0-d iteration errors
          top_k_indices = torch.topk(logits, k=k, dim=-1).indices.flatten().tolist()
          decoded_tokens = [tokenizer.decode([idx]).strip().lower() for idx in top_k_indices]

          if any(term in decoded_tokens for term in TASK_DESCRIPTIVE_TERMS):
            match_matrix[layer_idx, head_idx, prompt_idx] = True

    lexical_task_heads = []
    for l in range(n_layers):
      for h in range(n_heads):
        match_rate = match_matrix[l, h].float().mean().item()
        if match_rate >= p:
          lexical_task_heads.append({"layer": l, "head": h, "consistency": f"{match_rate*100:.1f}%"})

    return lexical_task_heads

In [83]:
lexical_task_heads = run_logit_lens_pipeline(inst_country_capital_eval_dataset)
print(f"Found {len(lexical_task_heads)} Lexical Task Heads.")
if lexical_task_heads:
    for lth in lexical_task_heads:
        print(f"Layer {lth['layer']}, Head {lth['head']} - Consistency: {lth['consistency']}")

Found 19 Lexical Task Heads.
Layer 1, Head 4 - Consistency: 50.0%
Layer 2, Head 8 - Consistency: 62.5%
Layer 2, Head 27 - Consistency: 75.0%
Layer 3, Head 7 - Consistency: 62.5%
Layer 9, Head 0 - Consistency: 50.0%
Layer 9, Head 4 - Consistency: 75.0%
Layer 9, Head 5 - Consistency: 75.0%
Layer 10, Head 3 - Consistency: 75.0%
Layer 11, Head 17 - Consistency: 62.5%
Layer 11, Head 22 - Consistency: 75.0%
Layer 12, Head 24 - Consistency: 75.0%
Layer 12, Head 27 - Consistency: 75.0%
Layer 12, Head 31 - Consistency: 75.0%
Layer 13, Head 1 - Consistency: 12.5%
Layer 13, Head 8 - Consistency: 75.0%
Layer 13, Head 20 - Consistency: 50.0%
Layer 13, Head 30 - Consistency: 75.0%
Layer 14, Head 22 - Consistency: 37.5%
Layer 15, Head 30 - Consistency: 50.0%


In [69]:
# Search for the LTHs in Llama-3.2-1B for the prompt in the paper
# The capital city of Japan is
target_item = inst_country_capital_eval_dataset[0]

cached_attn_inputs.clear()
inputs = tokenizer(target_item['prompt'], return_tensors="pt").to(device)
with torch.no_grad():
    model(**inputs)

found_any = False
for l in range(n_layers):
    for h in range(n_heads):
        single_head_input = cached_attn_inputs[l][0, h, :]
        logits = get_head_logits(model, tokenizer, l, h, single_head_input)
        top_k_indices = torch.topk(logits, k=5, dim=-1).indices.flatten().tolist()
        tokens = [tokenizer.decode([idx]).strip().lower() for idx in top_k_indices]

        if any(term in tokens for term in TASK_DESCRIPTIVE_TERMS):
            print(f"Found LTH at L{l+1} H{h}: {tokens}")
            found_any = True

if not found_any:
    print("No heads verbalized task terms for this specific prompt in the top 5 tokens.")

Found LTH at L4 H7: ['swagger', 'resh', 'urban', 'males', 'city']
Found LTH at L10 H4: ['cities', 'city', 'city', 'onia', 'city']
Found LTH at L10 H5: ['discretion', 'industrial', 'city', 'بزرگ', 'inos']
Found LTH at L11 H3: ['capital', 'счет', 'ancell', 'capital', 'sunk']
Found LTH at L12 H17: ['it', 'town', 'city', '"she', 'it']
Found LTH at L12 H22: ['capital', 'leaders', 'leader', 'leaders', 'capital']
Found LTH at L13 H24: ['citizens', 'citizen', 'city', 'cit', 'citizens']
Found LTH at L13 H27: ['city', 'cities', 'city', 'cities', 'city']
Found LTH at L13 H31: ['cap', 'capital', 'capital', 'capital', 'capital']
Found LTH at L14 H8: ['city', 'cities', '_city', 'dom', 'city']
Found LTH at L14 H20: ['響', 'fat', 'il', 'gu', 'capital']
Found LTH at L14 H30: ['city', 'capital', 'cities', 'infrastructure', 'urban']


In [84]:
# Identify LTHs for the Example-based (few-shot) style
lexical_task_heads_exam = run_logit_lens_pipeline(exam_country_capital_eval_dataset)

print(f"Found {len(lexical_task_heads_exam)} Lexical Task Heads in Example-based style.")

# Search for LTH for the prompt in the paper
# England: London; Korea: Seoul; Japan:
target_item = {'prompt': 'England: London; Korean: Seoul; Japan:',
  'target': 'Japan'}

cached_attn_inputs.clear()
inputs = tokenizer(target_item['prompt'], return_tensors="pt").to(device)
with torch.no_grad():
    model(**inputs)

found_any = False
for l in range(n_layers):
    for h in range(n_heads):
        single_head_input = cached_attn_inputs[l][0, h, :]
        logits = get_head_logits(model, tokenizer, l, h, single_head_input)
        top_k_indices = torch.topk(logits, k=5, dim=-1).indices.flatten().tolist()
        tokens = [tokenizer.decode([idx]).strip().lower() for idx in top_k_indices]

        if any(term in tokens for term in TASK_DESCRIPTIVE_TERMS):
            print(f"Found LTH at L{l+1} H{h}: {tokens}")
            found_any = True

if not found_any:
    print("No heads verbalized task terms for this specific prompt in the top 5 tokens.")

Found 9 Lexical Task Heads in Example-based style.
Found LTH at L10 H4: ['cities', 'city', 'sites', 'destinations', 'city']
Found LTH at L13 H24: ['city', 'citizens', 'citizen', 'ciudad', 'locals']
Found LTH at L13 H27: ['cities', 'city', 'country', 'countries', 'city']


In [86]:
inst_heads = set((l['layer'], l['head']) for l in lexical_task_heads)
exam_heads = set((l['layer'], l['head']) for l in lexical_task_heads_exam)
shared_heads = inst_heads.intersection(exam_heads)

print(f"\nShared Heads between Instruction and Example styles: {len(shared_heads)}")
for layer, head in sorted(shared_heads):
    # Calculate consistency for the instruction style for these shared heads
    inst_info = next(item for item in lexical_task_heads if item["layer"] == layer and item["head"] == head)
    exam_info = next(item for item in lexical_task_heads_exam if item["layer"] == layer and item["head"] == head)
    print(f"Layer {layer}, Head {head} | Inst: {inst_info['consistency']} | Exam: {exam_info['consistency']}")


Shared Heads between Instruction and Example styles: 7
Layer 9, Head 0 | Inst: 50.0% | Exam: 12.5%
Layer 9, Head 4 | Inst: 75.0% | Exam: 75.0%
Layer 9, Head 5 | Inst: 75.0% | Exam: 12.5%
Layer 11, Head 17 | Inst: 62.5% | Exam: 62.5%
Layer 12, Head 24 | Inst: 75.0% | Exam: 25.0%
Layer 12, Head 27 | Inst: 75.0% | Exam: 50.0%
Layer 13, Head 8 | Inst: 75.0% | Exam: 62.5%
